In [ ]:
import pandas as pd
import numpy as np
import struct
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import average_precision_score
from sklearn.preprocessing import LabelEncoder
import lightgbm as lgb
import warnings
warnings.filterwarnings('ignore')

## Load Data

In [ ]:
train = pd.read_csv('data/train.csv')
test = pd.read_csv('data/test.csv')
sub = pd.read_csv('data/sample_submission.csv')

CLASSES = ['Clutter', 'Cormorants', 'Pigeons', 'Ducks', 'Geese', 'Gulls', 'Birds of Prey', 'Waders', 'Songbirds']
print(f'Train: {len(train)}, Test: {len(test)}')
print(f'\nClass distribution:\n{train.bird_group.value_counts()}')

## Parse EWKB Trajectories

In [ ]:
def parse_ewkb_4d(hex_str):
    """Parse EWKB hex into list of (lon, lat, alt, rcs) tuples."""
    raw = bytes.fromhex(hex_str)
    offset = 0
    # byte order
    bo = '<' if raw[offset] == 1 else '>'
    offset += 1
    # geometry type (4 bytes) + SRID (4 bytes)
    geom_type = struct.unpack_from(f'{bo}I', raw, offset)[0]
    offset += 4
    # check if SRID flag is set (0x20000000)
    if geom_type & 0x20000000:
        offset += 4  # skip SRID
    # number of points
    n_points = struct.unpack_from(f'{bo}I', raw, offset)[0]
    offset += 4
    # each point: 4 doubles (lon, lat, alt, rcs)
    points = []
    for _ in range(n_points):
        lon, lat, alt, rcs = struct.unpack_from(f'{bo}4d', raw, offset)
        points.append((lon, lat, alt, rcs))
        offset += 32
    return points

# quick test
pts = parse_ewkb_4d(train.trajectory.iloc[0])
print(f'First track: {len(pts)} points')
print(f'Sample point (lon, lat, alt, rcs): {pts[0]}')

## Feature Engineering

In [ ]:
def haversine(lon1, lat1, lon2, lat2):
    """Haversine distance in meters between two points."""
    R = 6371000
    lon1, lat1, lon2, lat2 = map(np.radians, [lon1, lat1, lon2, lat2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2)**2
    return R * 2 * np.arcsin(np.sqrt(a))

def extract_trajectory_features(hex_str, traj_time_str):
    """Extract features from a single trajectory."""
    pts = parse_ewkb_4d(hex_str)
    times = eval(traj_time_str)  # list of floats
    
    lons = [p[0] for p in pts]
    lats = [p[1] for p in pts]
    alts = [p[2] for p in pts]
    rcs = [p[3] for p in pts]
    
    n = len(pts)
    
    # Segment distances
    dists = [haversine(lons[i], lats[i], lons[i+1], lats[i+1]) for i in range(n-1)] if n > 1 else [0]
    
    # Altitude changes
    alt_diffs = np.diff(alts) if n > 1 else [0]
    
    # Bearing changes (turning)
    bearings = []
    for i in range(n-1):
        dy = lats[i+1] - lats[i]
        dx = lons[i+1] - lons[i]
        bearings.append(np.arctan2(dy, dx))
    bearing_changes = np.diff(bearings) if len(bearings) > 1 else [0]
    # wrap to [-pi, pi]
    bearing_changes = np.arctan2(np.sin(bearing_changes), np.cos(bearing_changes))
    
    total_dist = sum(dists)
    duration = times[-1] - times[0] if len(times) > 1 else 0.001
    
    # Straight-line distance start to end
    straight_dist = haversine(lons[0], lats[0], lons[-1], lats[-1]) if n > 1 else 0
    sinuosity = total_dist / max(straight_dist, 1e-6)
    
    # Speeds between segments
    dt = np.diff(times) if len(times) > 1 else [1]
    dt = [max(d, 0.001) for d in dt]
    speeds = [d / t for d, t in zip(dists, dt)]
    
    feats = {
        'n_points': n,
        'duration': duration,
        'total_dist': total_dist,
        'straight_dist': straight_dist,
        'sinuosity': sinuosity,
        # altitude
        'alt_mean': np.mean(alts),
        'alt_std': np.std(alts),
        'alt_min': np.min(alts),
        'alt_max': np.max(alts),
        'alt_range': np.max(alts) - np.min(alts),
        'alt_diff_mean': np.mean(np.abs(alt_diffs)),
        # RCS
        'rcs_mean': np.mean(rcs),
        'rcs_std': np.std(rcs),
        'rcs_min': np.min(rcs),
        'rcs_max': np.max(rcs),
        'rcs_range': np.max(rcs) - np.min(rcs),
        'rcs_median': np.median(rcs),
        # speed
        'speed_mean': np.mean(speeds),
        'speed_std': np.std(speeds),
        'speed_max': np.max(speeds),
        'speed_min': np.min(speeds),
        'avg_ground_speed': total_dist / max(duration, 0.001),
        # turning
        'bearing_change_mean': np.mean(np.abs(bearing_changes)),
        'bearing_change_std': np.std(bearing_changes),
        'bearing_change_max': np.max(np.abs(bearing_changes)),
        # position
        'lon_mean': np.mean(lons),
        'lat_mean': np.mean(lats),
        'lon_std': np.std(lons),
        'lat_std': np.std(lats),
    }
    return feats

print('Extracting train features...')
train_feats = pd.DataFrame([extract_trajectory_features(r.trajectory, r.trajectory_time) for _, r in train.iterrows()])
print('Extracting test features...')
test_feats = pd.DataFrame([extract_trajectory_features(r.trajectory, r.trajectory_time) for _, r in test.iterrows()])
print(f'Features: {train_feats.shape[1]}')

In [ ]:
# Add non-trajectory features
for df_feat, df_orig in [(train_feats, train), (test_feats, test)]:
    # Timestamps
    ts = pd.to_datetime(df_orig['timestamp_start_radar_utc'])
    df_feat['hour'] = ts.dt.hour.values
    df_feat['month'] = ts.dt.month.values
    df_feat['dayofweek'] = ts.dt.dayofweek.values
    df_feat['minute'] = ts.dt.minute.values
    # hour as cyclic
    df_feat['hour_sin'] = np.sin(2 * np.pi * ts.dt.hour.values / 24)
    df_feat['hour_cos'] = np.cos(2 * np.pi * ts.dt.hour.values / 24)
    
    # Numeric cols
    df_feat['airspeed'] = df_orig['airspeed'].values
    df_feat['min_z'] = df_orig['min_z'].values
    df_feat['max_z'] = df_orig['max_z'].values
    df_feat['z_range'] = df_orig['max_z'].values - df_orig['min_z'].values
    
    # Radar bird size (categorical)
    size_map = {'Small bird': 0, 'Medium': 1, 'Large': 2, 'Flock': 3}
    df_feat['radar_bird_size'] = df_orig['radar_bird_size'].map(size_map).values

print(f'Final feature count: {train_feats.shape[1]}')
train_feats.head()

## Train LightGBM with 5-fold CV

In [ ]:
# Encode target
le = LabelEncoder()
le.fit(CLASSES)
y = le.transform(train['bird_group'])
n_classes = len(CLASSES)

X = train_feats.values
X_test = test_feats.values
feature_names = list(train_feats.columns)

# OOF predictions for CV score, test predictions averaged
oof_preds = np.zeros((len(X), n_classes))
test_preds = np.zeros((len(X_test), n_classes))

params = {
    'objective': 'multiclass',
    'num_class': n_classes,
    'metric': 'multi_logloss',
    'learning_rate': 0.05,
    'num_leaves': 31,
    'max_depth': 6,
    'min_child_samples': 10,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'reg_alpha': 0.1,
    'reg_lambda': 1.0,
    'verbose': -1,
    'seed': 42,
    'n_jobs': -1,
}

N_FOLDS = 5
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)

for fold, (tr_idx, va_idx) in enumerate(skf.split(X, y)):
    X_tr, X_va = X[tr_idx], X[va_idx]
    y_tr, y_va = y[tr_idx], y[va_idx]
    
    dtrain = lgb.Dataset(X_tr, label=y_tr, feature_name=feature_names)
    dval = lgb.Dataset(X_va, label=y_va, feature_name=feature_names, reference=dtrain)
    
    model = lgb.train(
        params,
        dtrain,
        num_boost_round=1000,
        valid_sets=[dval],
        callbacks=[lgb.early_stopping(50), lgb.log_evaluation(100)],
    )
    
    oof_preds[va_idx] = model.predict(X_va)
    test_preds += model.predict(X_test) / N_FOLDS
    
    # Per-fold mAP
    y_va_onehot = np.eye(n_classes)[y_va]
    fold_aps = []
    for c in range(n_classes):
        if y_va_onehot[:, c].sum() > 0:
            fold_aps.append(average_precision_score(y_va_onehot[:, c], oof_preds[va_idx][:, c]))
    print(f'Fold {fold}: mAP = {np.mean(fold_aps):.4f}')

# Overall CV mAP
y_onehot = np.eye(n_classes)[y]
aps = []
for c in range(n_classes):
    if y_onehot[:, c].sum() > 0:
        ap = average_precision_score(y_onehot[:, c], oof_preds[:, c])
        aps.append(ap)
        print(f'  {CLASSES[c]:15s}: AP = {ap:.4f}')
print(f'\nOverall CV mAP: {np.mean(aps):.4f}')

## Feature Importance

In [ ]:
imp = pd.DataFrame({'feature': feature_names, 'importance': model.feature_importance('gain')})
imp = imp.sort_values('importance', ascending=False)
print(imp.head(15).to_string(index=False))

## Generate Submission

In [ ]:
submission = pd.DataFrame({'track_id': test['track_id']})
for i, cls in enumerate(CLASSES):
    submission[cls] = test_preds[:, i]

submission.to_csv('submission.csv', index=False)
print(submission.head())
print(f'\nSaved submission.csv with {len(submission)} rows')